# Customer Churn Prediction - End-to-End Pipeline
This notebook builds a reusable scikit-learn pipeline to predict customer churn.
Steps: load data, preprocess (scaling/encoding), train Logistic Regression and Random Forest via GridSearchCV, evaluate and export the best pipeline with joblib.

In [ ]:
# 1. Imports
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

# For reproducibility
RANDOM_STATE = 42

In [ ]:
# 2. Load dataset (adjust filename if necessary)
DATA_PATH = 'cutomer_churn.csv'  # file in this notebook's folder - note the provided filename
assert os.path.exists(DATA_PATH), f"Dataset not found at {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print('Data shape:', df.shape)
df.head()

## Quick Cleaning and Feature/Target setup

In [ ]:
# 3. Basic cleaning
# Drop customerID since it's an identifier, not predictive
df = df.copy()
if 'customerID' in df.columns:
    df.drop(columns=['customerID'], inplace=True)

# TotalCharges may be read as object if there are blanks; coerce to numeric
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    # Fill missing TotalCharges with median to avoid chained-assignment warnings
    df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Target column: 'Churn' -> convert to binary 0/1
TARGET = 'Churn'
# Map Yes/No to 1/0 robustly for any dtype and cast to int
df[TARGET] = df[TARGET].replace({'Yes':1, 'No':0}).astype(int)

# Quick class balance check
print(df[TARGET].value_counts(normalize=True))

## Prepare feature lists for preprocessing

In [ ]:
# 4. Identify numeric and categorical columns automatically
# Exclude the target from features list if present in dtype selection
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].to_numpy()

# Force native pandas dtypes for safe indexing in environments using pyarrow-backed arrays
X = X.reset_index(drop=True)
# Convert object-like categoricals to string to avoid extension dtypes
for c in X.select_dtypes(include=['object','category']).columns:
    X[c] = X[c].astype(str)

numeric_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object','category','string']).columns.tolist()

print('Numeric cols:', numeric_cols)
print('Categorical cols (example):', categorical_cols[:10])

## Build preprocessing pipelines and full model pipeline

In [ ]:
# 5. Preprocessing for numeric and categorical features
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
)

# Define full pipeline with placeholder classifier to be set in GridSearchCV
pipe = Pipeline(steps=[('preprocessor', preprocessor), ('clf', LogisticRegression())])

## Train/Test split and GridSearchCV setup

In [ ]:
# 6. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

# 7. Grid search over two model families
param_grid = [
    {
        'clf': [LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)],
        "clf__C": [0.01, 0.1, 1, 10]
    },
    {
        'clf': [RandomForestClassifier(random_state=RANDOM_STATE)],
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 10, 20]
    }
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=2)

# Fit grid (this may take a few minutes depending on machine)
grid.fit(X_train, y_train)

print('Best score (cv):', grid.best_score_)
print('Best params:', grid.best_params_)

## Evaluation on Test Set

In [ ]:
# 8. Evaluate best estimator on test set
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:,1] if hasattr(best_model, 'predict_proba') else None

print('Classification report:')
print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
if y_proba is not None:
    print('Test ROC AUC:', roc_auc_score(y_test, y_proba))

## Export the best pipeline to disk with joblib

In [ ]:
# 9. Save best pipeline
OUTPUT_PATH = 'best_churn_pipeline.joblib'
joblib.dump(best_model, OUTPUT_PATH)
print('Saved best pipeline to', OUTPUT_PATH)

In [ ]:
# 10. Quick example: reload pipeline and predict on a few samples
loaded = joblib.load(OUTPUT_PATH)
sample = X_test.iloc[:5]
print('Predictions for first 5 test rows:', loaded.predict(sample))

: 

---
Notes: the GridSearchCV may take several minutes. You can reduce parameter grid sizes or use fewer CV folds to speed up.
If you want, I can also add a smaller example run or create a `requirements.txt` listing the main dependencies (`pandas`, `numpy`, `scikit-learn`, `joblib`).